In [5]:
### Implementing_Neural_Networks_with_PyTorch


import torch
import torch.nn as nn
import torchvision.transforms as transforms
from torch.utils.data import TensorDataset, DataLoader
import torch.optim as optim


## 데이터셋 생성
x_train = torch.rand(100, 10)
y_train = torch.rand(100, 1)

# 입력 데이터와 정답 레이블을 하나의 텐서 데이터셋 객체로 묶어준다
dataset = TensorDataset(x_train, y_train)

# 데이터셋을 로더에 올려 배치 분할 & 셔플 수행
dataloader = DataLoader(dataset, batch_size = 32, shuffle = True)


## 신경망 모델 정의
class SimpleNN(nn.Module):
  def __init__(self, input_size, hidden_size, output_size):
    super(SimpleNN, self).__init__()
    self.fc1 = nn.Linear(input_size, hidden_size)
    self.relu = nn.ReLU()
    self.fc2 = nn.Linear(hidden_size, output_size)
    # self.sigmoid = nn.Sigmoid() ← 시그모이드 함수를 층으로 정의할 때 대문자 S
  def forward(self, x):
    x = self.fc1(x)
    x = self.relu(x)
    x = self.fc2(x)
    # x = self.sigmoid(x)
    return x


## 모델 초기화
model = SimpleNN(input_size=10, hidden_size=20, output_size=1)
print(model)


## 손실 함수 및 옵티마이저 설정
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)


## 학습 루프 구현
for epoch in range(10):
  for batch in dataloader:
    inputs, targets = batch # 배치당 샘플 32개
    optimizer.zero_grad # 이전 기울기 0으로 초기화
    outputs = model(inputs) # 모델 입력
    loss = criterion(outputs, targets) # 손실 계산
    loss.backward() # 역전파를 통한 기울기 계산
    optimizer.step() # 계산된 기울기를 바탕으로 가중치 업데이트
  print(f"Epoch: {epoch+1}, Loss: {loss.item(): 4f}")


## 모델 저장 및 불러오기
# model.state_dict() & model.load_state_dict → nn.Module에서 상속받은 메서드
torch.save(model.state_dict(), 'model.pth')
model.load_state_dict(torch.load('model.pth'))


## 학습 중 NaN 값 확인 및 학습률 조정
if torch.isnan(loss).any():
  print("Loss contains NaN values! Reducing learning rate...")
  # optimizer.param_groups → 옵티마이저 최적화 대상 매개변수를 딕셔너리 형태로 저장
  for param_group in optimizer.param_groups:
    param_group['lr'] *= 0.1 # 딕셔너리에 저장된 각 매개변수 학습률 직접 축소


## 데이터 증강 기법 적용(이미지 데이터의 경우)
transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor()
])

SimpleNN(
  (fc1): Linear(in_features=10, out_features=20, bias=True)
  (relu): ReLU()
  (fc2): Linear(in_features=20, out_features=1, bias=True)
)
Epoch: 1, Loss:  0.301404
Epoch: 2, Loss:  0.103502
Epoch: 3, Loss:  0.167899
Epoch: 4, Loss:  0.298189
Epoch: 5, Loss:  0.070010
Epoch: 6, Loss:  0.143440
Epoch: 7, Loss:  0.056944
Epoch: 8, Loss:  0.106881
Epoch: 9, Loss:  0.096978
Epoch: 10, Loss:  0.069049
